# IEEE-CIS Fraud Detection — Exploratory Data Analysis

This notebook does an **initial pass** over the competition data and logs key findings to an MLflow experiment named `EDA`. Its goals are:

1. Confirm dataset shape and the train + identity merge.
2. Quantify class imbalance.
3. Inspect the temporal structure of `TransactionDT` (drives the CV strategy choice).
4. Find missing-heavy columns (drives cleaning strategy).
5. Identify high-cardinality categorical columns (drives encoding strategy).
6. Get a quick feature-importance ranking on a sub-sample (drives feature-selection strategy).

Nothing here trains a final model — it produces evidence we cite in the README and reuse when designing per-model experiments.

# 0. Setup

Paste the **Kaggle setup cell** below (from `notebooks/_kaggle_setup_cell.py`). It installs MLflow, pulls the `src/` package from GitHub, loads secrets, and configures imports.

> Edit the `REPO_URL` line in the snippet to point to your repo before running.

In [ ]:
# --- PASTE THE CONTENTS OF notebooks/_kaggle_setup_cell.py HERE ---
# (or, locally, do this minimal version:)
import os, sys
sys.path.insert(0, "..")  # so `from src import ...` works from notebooks/
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data import load_train, downcast_numerics
from src.feature_selection import (
    find_high_correlation, compute_mutual_info, compute_tree_importances,
)
from src.mlflow_utils import setup_mlflow, log_run, plot_feature_importance
import mlflow

# On Kaggle, `setup_mlflow()` will succeed because the secrets cell ran first.
# Locally without DagsHub configured, comment the next two lines out.
setup_mlflow("EDA")
print("MLflow tracking URI:", mlflow.get_tracking_uri())

# 1. Load and inspect

In [ ]:
DATA_PATH = "/kaggle/input/competitions/ieee-fraud-detection"

df = downcast_numerics(load_train(path=DATA_PATH))
# df = load_train()       # auto-detects /kaggle/input or local
df = downcast_numerics(df)
print(f"shape : {df.shape}")
print(f"memory: {df.memory_usage(deep=False).sum() / 1024**2:,.0f} MB")
df.head(3)

# 2. Class balance

Fraud (`isFraud=1`) is rare. We need to log the exact ratio because it influences:
- which `class_weight` we use,
- whether we add a `scale_pos_weight` to XGBoost,
- whether ROC-AUC remains a sane metric (it does; AUC is invariant to class prior).

In [ ]:
y = df["isFraud"]
n_pos = int(y.sum())
n_neg = int((y == 0).sum())
pos_rate = n_pos / len(y)
print(f"fraud (1) : {n_pos:>7,}  ({pos_rate:6.3%})")
print(f"legit (0) : {n_neg:>7,}  ({1-pos_rate:6.3%})")
print(f"pos:neg   ≈ 1 : {n_neg / n_pos:.1f}  (use this for scale_pos_weight)")

# 3. Temporal structure of `TransactionDT`

`TransactionDT` is "seconds since some reference time". The crucial observation: **train and public test do not overlap in time** (Kaggle's competition setup). That means:

- A random K-Fold leaks the future into training → AUC is over-optimistic.
- `TimeSeriesSplit` better mirrors the public LB.
- The README **must** call this out.

The plot below shows the distribution; it should look monotone in time.

In [ ]:
dt_min, dt_max = df["TransactionDT"].min(), df["TransactionDT"].max()
print(f"TransactionDT  range: {dt_min:>10,}  ..  {dt_max:>10,}")
print(f"span (days)        : {(dt_max - dt_min) / 86400:.1f}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(df["TransactionDT"], bins=120)
ax.set_title("Transactions over time (train)")
ax.set_xlabel("TransactionDT (seconds)")
plt.show()

# 4. Missingness

The dataset has many columns where the vast majority of values are NaN. These are usually safe to drop. The threshold we pick here (90% / 95% / 99%) becomes one of the **cleaning experiments** for every model.

In [ ]:
nan_frac = df.isna().mean().sort_values(ascending=False)

for thr in (0.50, 0.75, 0.90, 0.95, 0.99):
    n = int((nan_frac > thr).sum())
    print(f"  cols with NaN > {thr:>4.0%}: {n:>3} / {df.shape[1]}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(nan_frac, bins=60)
ax.set_title("Distribution of NaN fraction across columns")
ax.set_xlabel("NaN fraction"); ax.set_ylabel("number of columns")
plt.show()

# 5. Categorical cardinality

Columns like `card1` have ~13k unique levels. One-hot would explode them, so we either:
- frequency-encode them (cheap, surprisingly strong),
- target-encode them (with smoothing — but careful with leakage),
- group rare categories and one-hot the rest.

For trees this matters less; for Logistic Regression and the MLP it matters a lot.

In [ ]:
cat_candidates = (
    df.select_dtypes(include=["object"]).columns.tolist()
    + ["card1", "card2", "card3", "card5", "addr1", "addr2"]
)
cat_candidates = [c for c in cat_candidates if c in df.columns]

card_summary = (
    pd.DataFrame({
        "n_unique": [df[c].nunique(dropna=True) for c in cat_candidates],
        "n_nan":    [int(df[c].isna().sum())   for c in cat_candidates],
    }, index=cat_candidates)
    .sort_values("n_unique", ascending=False)
)
card_summary.head(20)

# 6. `TransactionAmt` distribution

Heavy right-tail — log1p makes Logistic Regression and the MLP much happier; trees don't care.

In [ ]:
amt = df["TransactionAmt"]
print(amt.describe(percentiles=[.5, .9, .99, .999]).round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(amt, bins=80); axes[0].set_title("TransactionAmt (raw)")
axes[1].hist(np.log1p(amt), bins=80); axes[1].set_title("log1p(TransactionAmt)")
plt.show()

# 7. Quick feature ranking on a 100k sample

We fit a small Random Forest as a feature-ranking probe (NOT a final model). The output guides the **top-K** lists used in `feature_selection`.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

sample = df.sample(min(100_000, len(df)), random_state=42)
y_s = sample["isFraud"]
X_s = sample.drop(columns=["isFraud"])

rf = RandomForestClassifier(
    n_estimators=80, max_depth=12, n_jobs=-1, random_state=42,
    class_weight="balanced",
)
imp = compute_tree_importances(rf, X_s, y_s)
imp.head(30)

In [ ]:
fig = plot_feature_importance(imp.index, imp.values, top=30,
                              title="Top 30 by RF importance (100k sample)")
plt.show()

# 8. Log everything as one MLflow `EDA` run

We log the dataset shape, class balance, missingness summary and top-30 feature importance plot as **artifacts** so the README can link straight to them.

In [ ]:
with log_run(
    run_name="EDA_overview",
    tags={"stage": "eda", "purpose": "data overview"},
    params={
        "n_rows": int(len(df)),
        "n_cols": int(df.shape[1]),
        "missing_threshold_candidates": "0.5 / 0.75 / 0.9 / 0.95 / 0.99",
    },
):
    mlflow.log_metric("pos_rate",   float(pos_rate))
    mlflow.log_metric("dt_min",     float(dt_min))
    mlflow.log_metric("dt_max",     float(dt_max))
    mlflow.log_metric("span_days",  float((dt_max - dt_min) / 86400))
    mlflow.log_metric("nan_gt_50pct", int((nan_frac > 0.50).sum()))
    mlflow.log_metric("nan_gt_75pct", int((nan_frac > 0.75).sum()))
    mlflow.log_metric("nan_gt_90pct", int((nan_frac > 0.90).sum()))
    mlflow.log_metric("nan_gt_95pct", int((nan_frac > 0.95).sum()))
    mlflow.log_metric("nan_gt_99pct", int((nan_frac > 0.99).sum()))
    mlflow.log_figure(fig, "rf_top30_importances.png")

    # Save top-100 by importance as a plain text artifact
    top100 = imp.head(100).index.tolist()
    with open("eda_top100_features.txt", "w") as f:
        f.write("\n".join(top100))
    mlflow.log_artifact("eda_top100_features.txt")
    print("Logged EDA run.")